In [1]:
import preprocessamento as pp
import treinamento as tr

import pandas as pd
import yfinance as yf

In [2]:
from datetime import datetime, timedelta

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score
from sklearn.metrics import precision_recall_curve
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# Parâmetros
DIAS_VALIDACAO = 90  # número de dias para simular produção
base = datetime.today().date() - timedelta(days=1)

In [5]:
df_experimentos = pd.DataFrame()

In [6]:
lista_tickers = ['ABEV3.SA', 'AZUL4.SA', 'B3SA3.SA', 'BBAS3.SA', 'BBDC3.SA', 'BBDC4.SA', 'BBSE3.SA', 'BEEF3.SA', 'BPAC11.SA', 'BRAP4.SA', 'BRDT3.SA', 'BRFS3.SA', 'BRKM5.SA', 'BRML3.SA', 'BTOW3.SA', 'CCRO3.SA', 'CIEL3.SA', 'CMIG4.SA', 'COGN3.SA', 'CPFE3.SA', 'CRFB3.SA', 'CSAN3.SA', 'CSNA3.SA', 'CVCB3.SA', 'CYRE3.SA', 'ECOR3.SA', 'EGIE3.SA', 'ELET3.SA', 'ELET6.SA', 'EMBR3.SA', 'ENBR3.SA', 'ENGI11.SA', 'EQTL3.SA', 'EZTC3.SA', 'FLRY3.SA', 'GGBR4.SA', 'GNDI3.SA', 'GOAU4.SA', 'HAPV3.SA', 'HGTX3.SA', 'HYPE3.SA', 'IGTA3.SA', 'IRBR3.SA', 'ITSA4.SA', 'ITUB4.SA', 'JBSS3.SA', 'KLBN11.SA', 'LAME4.SA', 'LREN3.SA', 'MGLU3.SA', 'MRFG3.SA', 'MRVE3.SA', 'MULT3.SA', 'NTCO3.SA', 'PCAR3.SA', 'PETR3.SA', 'PETR4.SA', 'PRIO3.SA', 'QUAL3.SA', 'RADL3.SA', 'RAIL3.SA', 'RENT3.SA', 'SANB11.SA', 'SBSP3.SA', 'SULA11.SA', 'SUZB3.SA', 'TAEE11.SA', 'TIMS3.SA', 'TOTS3.SA', 'UGPA3.SA', 'USIM5.SA', 'VALE3.SA', 'VIVT4.SA', 'VVAR3.SA', 'WEGE3.SA', 'YDUQ3.SA']
#lista_tickers = ['HAPV3.SA', 'HGTX3.SA', 'HYPE3.SA', 'IGTA3.SA', 'IRBR3.SA', 'ITSA4.SA', 'ITUB4.SA', 'JBSS3.SA', 'KLBN11.SA', 'LAME4.SA', 'LREN3.SA', 'MGLU3.SA', 'MRFG3.SA', 'MRVE3.SA', 'MULT3.SA', 'NTCO3.SA', 'PCAR3.SA', 'PETR3.SA', 'PETR4.SA', 'PRIO3.SA', 'QUAL3.SA', 'RADL3.SA', 'RAIL3.SA', 'RENT3.SA', 'SANB11.SA', 'SBSP3.SA', 'SULA11.SA', 'SUZB3.SA', 'TAEE11.SA', 'TIMS3.SA', 'TOTS3.SA', 'UGPA3.SA', 'USIM5.SA', 'VALE3.SA', 'VIVT4.SA', 'VVAR3.SA', 'WEGE3.SA', 'YDUQ3.SA']
lista_janelas = [6, 10, 15, 30, 60]

In [7]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.base import clone # Para criar novas instâncias do modelo

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit # <<< A estrela da função

def avaliar_estrategia_com_timeseriessplit(
        modelo_template, # Um modelo NÃO treinado
        X_super_treino: pd.DataFrame,
        y_super_treino: pd.Series,
        n_splits: int = 5 # <<< Novo parâmetro principal: número de folds
    ):
    """
    Avalia uma estratégia de modelagem usando TimeSeriesSplit do scikit-learn
    DENTRO do conjunto de "super treino". O modelo é retreinado em cada split.

    Args:
        modelo_template: Uma instância NÃO TREINADA do modelo a ser avaliado.
        X_super_treino: O conjunto completo de features de treino.
        y_super_treino: Os alvos completos de treino.
        n_splits (int): O número de splits (folds) a serem criados.

    Returns:
        dict: Dicionário com as métricas médias (f1_medio, precisao_media, recall_medio).
    """
    print(f"\n--- Avaliando Estratégia com TimeSeriesSplit (n_splits={n_splits}) ---")

    # TimeSeriesSplit cria janelas de treino que se expandem.
    # Ex: para n_splits=5:
    # Fold 1: treina em [bloco 1], testa em [bloco 2]
    # Fold 2: treina em [bloco 1, bloco 2], testa em [bloco 3]
    # Fold 3: treina em [bloco 1, bloco 2, bloco 3], testa em [bloco 4]
    # ... e assim por diante.
    tscv = TimeSeriesSplit(n_splits=n_splits)

    f1s, precisions, recalls = [], [], []

    # O loop agora itera sobre os splits gerados pelo tscv
    for fold, (train_index, test_index) in enumerate(tscv.split(X_super_treino)):
        X_treino_fold = X_super_treino.iloc[train_index]
        y_treino_fold = y_super_treino.iloc[train_index]
        X_teste_fold = X_super_treino.iloc[test_index]
        y_teste_fold = y_super_treino.iloc[test_index]

        print(f"Fold {fold + 1}/{n_splits}: Treinando com {len(X_treino_fold)} amostras, Validando com {len(X_teste_fold)} amostras.")

        # Retreinar o modelo em cada fold de treino
        modelo_fold = clone(modelo_template)
        try:
            modelo_fold.fit(X_treino_fold.values, y_treino_fold.values) # Usando .values por consistência
        except Exception as e:
            print(f"  ERRO no fit do Fold {fold + 1}: {e}. Pulando este fold.")
            continue

        # Fazer previsões (threshold padrão de 0.5)
        y_pred_fold = modelo_fold.predict(X_teste_fold.values)

        # Calcular e armazenar métricas
        f1s.append(f1_score(y_teste_fold, y_pred_fold, zero_division=0))
        precisions.append(precision_score(y_teste_fold, y_pred_fold, zero_division=0))
        recalls.append(recall_score(y_teste_fold, y_pred_fold, zero_division=0))

    if not f1s: # Se nenhuma janela foi processada com sucesso
        print("AVISO: Nenhuma métrica foi calculada. Verifique os dados e os splits.")
        return {'f1_medio': 0, 'precisao_media': 0, 'recall_medio': 0}

    # Calcular as médias das métricas de todos os folds
    resultado = {
        'f1_medio': np.mean(f1s),
        'f1_std': np.std(f1s),
        'precisao_media': np.mean(precisions),
        'recall_medio': np.mean(recalls)
    }

    print(f"Resultado Médio dos {len(f1s)} Folds: F1={resultado['f1_medio']:.3f}, Precisão={resultado['precisao_media']:.3f}, Recall={resultado['recall_medio']:.3f}")
    return resultado

In [9]:
i = 0
df_experimentos = pd.DataFrame()
lista_janelas = [6, 10, 15, 20, 30]
#lista_tickers = ["CSAN3", "PETR4", "VALE3"]
#base = date.today()
DIAS_VALIDACAO = 90

for papel in lista_tickers:    
    try:
        # 1. Obter e preparar os dados e o target
        # ... (sua lógica para determinar valorizacao_sugerida e dias_futuro_final) ...
        df_base = pp.montar_dataset(papel, dias_total_historico=700, data_base=base)
        # ...
        ultimo_atr_pct = df_base['atr_pct'].iloc[-1]
        multiplo_atr = 1.5
        valorizacao_sugerida = ultimo_atr_pct * multiplo_atr
        # arredondar valorização sugerida para cima com duas casas
        valorizacao_sugerida = np.ceil(valorizacao_sugerida * 100) / 100
        stop = np.ceil(ultimo_atr_pct)  # stop loss sugerido como 1 x último ATR

        for d in range(1, 11):
            print(f'Simulando {d} dias de valorização futura...')
            dias_futuro = d
            df = pp.calcular_target_com_stop(df_base, valorizacao_alvo_pct=valorizacao_sugerida, dias_horizonte_alvo=d, stop_loss_pct=stop)
            
            if (df['Target'].value_counts(normalize=True).values[1] > 0.2):            
                print(f'Valorização futura de {dias_futuro} dias é viável.')
                break

        dias_futuro_final = d # Exemplo        
        df = pp.calcular_target_com_stop(df_base, valorizacao_alvo_pct=valorizacao_sugerida, dias_horizonte_alvo=dias_futuro_final, stop_loss_pct=stop)
    except Exception as e:
        print(f'Erro na preparação de dados para {papel}: {e}')
        continue
    
    for dias_janela_x in lista_janelas:    
        print(f'\nAnalisando {papel} com janela de {dias_janela_x} dias...')    
        
        X, y = pp.preparar_X_y_agregado(df, dias_janela_x, dropna=True, tipo_drop_na='linhas')    
        print(X.head())
        if len(X) < 150: # Mínimo para TimeSeriesSplit(n_splits=5) ter um treino inicial razoável
            print(f"Dados insuficientes ({len(X)} linhas) após preparação. Pulando config.")
            continue
            
        X_train, y_train, X_validacao, y_validacao = tr.dividir_treino_validacao(X, y, dias_validacao=DIAS_VALIDACAO, dias_futuro=dias_futuro_final)

        # Modelo template - será clonado dentro da função de avaliação
        modelo_template = LogisticRegression(solver='liblinear', penalty='l2', random_state=42, class_weight='balanced')
        
        # --- AVALIAÇÃO ROBUSTA COM TimeSeriesSplit DENTRO DO CONJUNTO DE TREINO ---
        metricas_wf_treino = avaliar_estrategia_com_timeseriessplit(
            modelo_template=modelo_template,
            X_super_treino=X_train,
            y_super_treino=y_train,
            n_splits=5 # <<< Definindo o número de folds desejado
        )
        
        f1_wf_treino = metricas_wf_treino['f1_medio']
        f1_wf_treino_std = metricas_wf_treino['f1_std']
        
        
        # --- LOG DOS EXPERIMENTOS ---
        i += 1
        experimento_atual = {
            'papel': papel,
            'dias_futuro': dias_futuro_final,
            'dias_janela_x': dias_janela_x,
            # 'ultimo_atr_pct': ultimo_atr_pct, # Adicione se precisar
            'valorizacao': valorizacao_sugerida,
            'stop': stop,
            'f1_wf_treino': f1_wf_treino,
            'f1_wf_treino_std': f1_wf_treino_std,
            'precisao_wf_treino': metricas_wf_treino['precisao_media'],
            'recall_wf_treino': metricas_wf_treino['recall_medio']
        }
        df_experimentos = pd.concat([df_experimentos, pd.DataFrame([experimento_atual], index=[i])])

[ABEV3.SA] Cache existente cobre de 2020-01-02 a 2025-07-31.
[ABEV3.SA] Cache cobre período solicitado (2022-09-14 a 2025-07-30).
[ABEV3.SA] Garantindo atualização dos últimos 5 dias do cache...
Index(['data', 'adj_close', 'close', 'Dividends', 'high', 'low', 'open',
       'Stock Splits', 'volume'],
      dtype='object', name='Price')
[ABEV3.SA] Últimos dias atualizados em memória com sucesso.
[ABEV3.SA] Retornando 717 linhas para o período solicitado.
✅ Usando cache local: /palpiteiro/dados\macro_20210914_20250731.csv
[^BVSP] Cache existente cobre de 2020-01-02 a 2025-07-31.
[^BVSP] Cache cobre período solicitado (2022-09-14 a 2025-07-30).
[^BVSP] Garantindo atualização dos últimos 5 dias do cache...
Index(['data', 'adj_close', 'close', 'Dividends', 'high', 'low', 'open',
       'Stock Splits', 'volume'],
      dtype='object', name='Price')
[^BVSP] Últimos dias atualizados em memória com sucesso.
[^BVSP] Retornando 717 linhas para o período solicitado.
[BOVA11.SA] Cache existente cob

KeyboardInterrupt: 

In [ ]:
df_experimentos.to_csv('experimentos_wf.csv')

In [ ]:
df_experimentos.sort_values(by='f1_wf_treino', ascending=False).head(10)

,papel,dias_futuro,dias_janela_x,valorizacao,stop,f1_wf_treino,f1_wf_treino_std,precisao_wf_treino,recall_wf_treino
76,CRFB3.SA,1,6,0.02,1.0,0.503413,0.044723,0.384116,0.739385
79,CRFB3.SA,1,20,0.02,1.0,0.470250,0.050407,0.362494,0.670220
78,CRFB3.SA,1,15,0.02,1.0,0.436249,0.069939,0.359104,0.599074
80,CRFB3.SA,1,30,0.02,1.0,0.418495,0.101136,0.335502,0.581753
92,CVCB3.SA,3,10,0.08,1.0,0.395764,0.123343,0.324121,0.561508
127,ENGI11.SA,3,10,0.04,1.0,0.378653,0.153275,0.277544,0.723262
170,ITUB4.SA,3,30,0.03,1.0,0.375152,0.085091,0.258594,0.764502
54,BRKM5.SA,5,20,0.07,1.0,0.372261,0.063556,0.315282,0.620980
175,KLBN11.SA,5,30,0.04,1.0,0.370815,0.128119,0.258117,0.686993
126,ENGI11.SA,3,6,0.04,1.0,0.358928,0.119174,0.276227,0.666291


In [ ]:
#df_experimentos.to_csv('experimentos_v2.csv', index=False)

In [4]:
df = pd.read_csv('experimentos_v2.csv')
df.head()

,papel,dias_futuro,dias_janela_x,ultimo_atr_pct,valorizacao,roc_treino,roc_validacao,diferenca,pr_auc_treino,pr_auc_validacao,f1_treino,f1_validacao
0,ABEV3.SA,4.0,6.0,0.019791,0.03,0.637019,0.439777,0.197243,0.421820,0.413704,0.291677,0.123612
1,ABEV3.SA,4.0,10.0,0.019791,0.03,0.522723,0.408518,0.114205,0.412751,0.480152,0.288502,0.288148
2,ABEV3.SA,4.0,15.0,0.019791,0.03,0.572263,0.742435,-0.170172,0.365960,0.624386,0.318050,0.429850
3,ABEV3.SA,4.0,30.0,0.019791,0.03,0.592138,0.235878,0.356260,0.387937,0.334228,0.269833,0.199938
4,ABEV3.SA,4.0,60.0,0.019791,0.03,0.658336,0.446249,0.212088,0.415611,0.459915,0.342372,0.356601


In [ ]:
#df_2 = pd.read_csv('experimentos_2.csv')

In [ ]:
df_geral = pd.concat([df], ignore_index=True)
df_geral.head()

,papel,dias_futuro,dias_janela_x,ultimo_atr_pct,valorizacao,roc_treino,roc_validacao,diferenca,pr_auc_treino,pr_auc_validacao,f1_treino,f1_validacao
0,ABEV3.SA,4.0,6.0,0.019791,0.03,0.637019,0.439777,0.197243,0.421820,0.413704,0.291677,0.123612
1,ABEV3.SA,4.0,10.0,0.019791,0.03,0.522723,0.408518,0.114205,0.412751,0.480152,0.288502,0.288148
2,ABEV3.SA,4.0,15.0,0.019791,0.03,0.572263,0.742435,-0.170172,0.365960,0.624386,0.318050,0.429850
3,ABEV3.SA,4.0,30.0,0.019791,0.03,0.592138,0.235878,0.356260,0.387937,0.334228,0.269833,0.199938
4,ABEV3.SA,4.0,60.0,0.019791,0.03,0.658336,0.446249,0.212088,0.415611,0.459915,0.342372,0.356601


In [ ]:
df_geral['dif_f1'] = df_geral['f1_validacao'] - df_geral['f1_treino']

In [ ]:
df_geral[df_geral['dif_f1']<0.1].sort_values(by=['f1_validacao'], ascending=[False]).head(10)

,papel,dias_futuro,dias_janela_x,ultimo_atr_pct,valorizacao,roc_treino,roc_validacao,diferenca,pr_auc_treino,pr_auc_validacao,f1_treino,f1_validacao,dif_f1
163,ITSA4.SA,4.0,30.0,0.017870,0.03,0.670695,0.599028,0.071667,0.422011,0.584732,0.394375,0.477366,0.082991
83,CSAN3.SA,8.0,30.0,0.040990,0.07,0.649047,0.519109,0.129938,0.469296,0.561376,0.391274,0.462760,0.071486
80,CSAN3.SA,8.0,6.0,0.040990,0.07,0.696752,0.477393,0.219360,0.488576,0.525477,0.419898,0.446821,0.026923
289,WEGE3.SA,7.0,60.0,0.026813,0.05,0.619024,0.644886,-0.025862,0.407603,0.471731,0.393868,0.434428,0.040560
32,BEEF3.SA,4.0,15.0,0.037846,0.06,0.681190,0.760556,-0.079366,0.415242,0.729839,0.384723,0.430998,0.046276
212,PETR4.SA,3.0,15.0,0.019530,0.03,0.491795,0.582772,-0.090977,0.405946,0.395777,0.402071,0.421848,0.019776
198,MULT3.SA,4.0,30.0,0.022971,0.04,0.665363,0.550814,0.114549,0.451190,0.512980,0.332724,0.420615,0.087892
36,BPAC11.SA,3.0,10.0,0.024858,0.04,0.591748,0.672833,-0.081085,0.399926,0.456177,0.367266,0.420289,0.053023
31,BEEF3.SA,4.0,10.0,0.037846,0.06,0.586611,0.618833,-0.032222,0.392949,0.425468,0.343096,0.416715,0.073619
150,GOAU4.SA,4.0,6.0,0.024946,0.04,0.593185,0.614496,-0.021311,0.331318,0.484376,0.338087,0.411299,0.073212


In [ ]:
df_geral.sort_values(by=['pr_auc_validacao'], ascending=[False]).head(30)

,papel,dias_futuro,dias_janela_x,ultimo_atr_pct,valorizacao,roc_treino,roc_validacao,diferenca,pr_auc_treino,pr_auc_validacao,f1_treino,f1_validacao
37,BEEF3.SA,8.0,15.0,0.045873,0.07,0.785121,0.710152,0.074968,0.551479,0.825363,0.445723,0.784262
14,B3SA3.SA,6.0,60.0,0.029865,0.05,0.840791,0.844461,-0.003670,0.642239,0.794473,0.518558,0.441595
205,MRFG3.SA,5.0,6.0,0.040430,0.07,0.779585,0.826902,-0.047317,0.588089,0.787441,0.504083,0.573671
147,FLRY3.SA,7.0,15.0,0.024547,0.04,0.767655,0.882989,-0.115333,0.597972,0.782418,0.532214,0.522342
264,RENT3.SA,5.0,60.0,0.032095,0.05,0.850567,0.790949,0.059618,0.587925,0.779796,0.461715,0.743000
39,BEEF3.SA,8.0,60.0,0.045873,0.07,0.833054,0.739486,0.093568,0.605884,0.772826,0.512033,0.392641
206,MRFG3.SA,5.0,10.0,0.040430,0.07,0.795152,0.788248,0.006903,0.594322,0.771853,0.548604,0.384880
310,WEGE3.SA,9.0,6.0,0.027673,0.05,0.754549,0.821674,-0.067126,0.572965,0.771779,0.469297,0.593440
36,BEEF3.SA,8.0,10.0,0.045873,0.07,0.742462,0.702689,0.039773,0.525735,0.770377,0.446930,0.749333
148,FLRY3.SA,7.0,30.0,0.024547,0.04,0.773159,0.881200,-0.108042,0.587808,0.760693,0.516747,0.494774


In [ ]:



# --- SEU LOOP PRINCIPAL DE SCREENING, AGORA MODIFICADO ---




# --- ANÁLISE PÓS-SCREENING ---
# Agora você ordena seus resultados com base em uma métrica robusta, não enviesada.
print("\n--- Tabela Final de Experimentos ---")
df_experimentos_ordenados = df_experimentos.sort_values(by='f1_wf_treino', ascending=False)
print(df_experimentos_ordenados.to_string())

# Agora, para o seu melhor candidato da lista (ex: a primeira linha),
# você pode finalmente usar o conjunto de validação (hold-out) para
# ter uma estimativa final e imparcial do desempenho.
if not df_experimentos_ordenados.empty:
    melhor_config = df_experimentos_ordenados.iloc[0]
    print(f"\n--- Avaliação Final do Melhor Candidato no Hold-out ---")
    print(f"Configuração: Papel={melhor_config['papel']}, Janela X={melhor_config['dias_janela_x']} dias")
    
    # Recriar os dados para a melhor configuração
    df_base_final = pp.montar_dataset(melhor_config['papel'], dias_total_historico=700, data_base=base)
    df_final = pp.calcular_target_com_stop(df_base_final, valorizacao_alvo_pct=melhor_config['valorizacao'], dias_horizonte_alvo=melhor_config['dias_futuro'], stop_loss_pct=melhor_config['stop'])
    X_final, y_final = pp.preparar_X_y_agregado(df_final, int(melhor_config['dias_janela_x']), dropna=True)    
    X_train_final, y_train_final, X_validacao_final, y_validacao_final = tr.dividir_treino_validacao(X_final, y_final, DIAS_VALIDACAO, dias_futuro=melhor_config['dias_futuro'])

    # Treinar o modelo final com todo o conjunto de treino
    modelo_final = LogisticRegression(solver='liblinear', penalty='l2', random_state=42, class_weight='balanced')
    modelo_final.fit(X_train_final.values, y_train_final.values)

    # Avaliar no hold-out
    y_pred_validacao_final = modelo_final.predict(X_validacao_final.values)
    f1_holdout = f1_score(y_validacao_final, y_pred_validacao_final)
    prec_holdout = precision_score(y_validacao_final, y_pred_validacao_final)
    rec_holdout = recall_score(y_validacao_final, y_pred_validacao_final)

    print(f"Desempenho no Hold-Out: F1={f1_holdout:.3f}, Precisão={prec_holdout:.3f}, Recall={rec_holdout:.3f}")

In [10]:
def calcular_recall_em_precisao_minima(y_true, y_probas, precisao_minima=0.50):
    """
    (Função Auxiliar)
    Calcula o maior recall possível para uma precisão mínima desejada.

    Args:
        y_true: Rótulos verdadeiros.
        y_probas: Probabilidades previstas para a classe positiva.
        precisao_minima (float): O nível de precisão mínimo desejado.

    Returns:
        float: O valor do recall no ponto ótimo, ou 0 se a precisão mínima não for atingida.
    """
    # Se não houver nenhum evento positivo real, o recall é indefinido. Retornamos 0.
    if np.sum(y_true) == 0:
        return 0.0

    # precision_recall_curve retorna precisões, recalls e thresholds correspondentes
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probas)

    # A curva retorna um valor a mais para precisão e recall do que para thresholds.
    # Excluímos o último ponto (recall=0, precision=1) que é um artefato da função.
    precisions = precisions[:-1]
    recalls = recalls[:-1]

    # Encontrar os índices onde a precisão é >= à mínima desejada
    indices_validos = np.where(precisions >= precisao_minima)[0]

    if len(indices_validos) == 0:
        # Se nenhum ponto na curva atinge a precisão mínima, o recall efetivo é 0.
        return 0.0
    else:
        # Dentre os pontos que atendem à precisão, pegamos o que tem o maior recall.
        melhor_recall = np.max(recalls[indices_validos])
        return melhor_recall


def avaliar_estrategia_com_recall_em_precisao_minima(
        modelo_template,
        X_super_treino: pd.DataFrame,
        y_super_treino: pd.Series,
        n_splits: int = 5,
        precisao_minima_desejada: float = 0.60
    ):
    """
    Avalia uma estratégia de modelagem usando TimeSeriesSplit, focando na métrica
    "Recall na Precisão Mínima".

    Retorna métricas médias e desvio padrão para o screening.
    """
    print(f"\n--- Avaliando Estratégia com Recall na Precisão Mínima de {precisao_minima_desejada:.0%} (n_splits={n_splits}) ---")
    
    tscv = TimeSeriesSplit(n_splits=n_splits)
    recalls_na_precisao_minima = []

    for fold, (train_index, test_index) in enumerate(tscv.split(X_super_treino)):
        X_treino_fold = X_super_treino.iloc[train_index]
        y_treino_fold = y_super_treino.iloc[train_index] 

        # validar nos 30 dias seguintes       
        X_teste_fold = X_super_treino.iloc[test_index[:30]]
        y_teste_fold = y_super_treino.iloc[test_index[:30]]

        print(f"Fold {fold + 1}/{n_splits}: Treinando com {len(X_treino_fold)}, Validando com {len(X_teste_fold)}...")

        modelo_fold = clone(modelo_template)
        try:
            modelo_fold.fit(X_treino_fold, y_treino_fold)
        except Exception as e:
            print(f"  ERRO no fit do Fold {fold + 1}: {e}. Pulando.")
            recalls_na_precisao_minima.append(0) # Adiciona 0 se o treino falhar
            continue
        
        # Obter probabilidades no conjunto de teste do fold
        probas_teste_fold = modelo_fold.predict_proba(X_teste_fold)[:, 1]

        # Calcular a nossa métrica customizada para este fold
        recall_neste_fold = calcular_recall_em_precisao_minima(y_teste_fold, probas_teste_fold, precisao_minima_desejada)
        recalls_na_precisao_minima.append(recall_neste_fold)
        print(f"  Recall na Precisão >= {precisao_minima_desejada:.0%}: {recall_neste_fold:.3f}")

    if not recalls_na_precisao_minima:
        return {'recall_medio_na_prec_min': 0, 'recall_std_na_prec_min': 0, 'prob_sucesso_folds': 0}

    # Calcular as estatísticas finais
    recall_medio = np.mean(recalls_na_precisao_minima)
    recall_std = np.std(recalls_na_precisao_minima)
    # Probabilidade de sucesso: % de folds onde o recall foi > 0
    prob_sucesso_folds = np.mean([1 if r > 0 else 0 for r in recalls_na_precisao_minima])

    print(f"\nResultado Final da Avaliação Walk-Forward:")
    print(f"  Recall Médio (com Precisão >= {precisao_minima_desejada:.0%}): {recall_medio:.3f}")
    print(f"  Desvio Padrão do Recall: {recall_std:.3f}")
    print(f"  Percentual de Folds com Sinal Viável: {prob_sucesso_folds:.0%}")

    resultado = {
        'recall_medio_na_prec_min': recall_medio,
        'recall_std_na_prec_min': recall_std,
        'prob_sucesso_folds': prob_sucesso_folds
    }
    return resultado

In [11]:
i = 0
df_experimentos = pd.DataFrame()
lista_janelas = [6, 10, 15, 20, 30]
DIAS_VALIDACAO = 90

for papel in lista_tickers:    
    try:
        # 1. Obter e preparar os dados e o target
        # ... (sua lógica para determinar valorizacao_sugerida e dias_futuro_final) ...
        df_base = pp.montar_dataset(papel, dias_total_historico=700, data_base=base)
        # ...
        ultimo_atr_pct = df_base['atr_pct'].iloc[-1]
        multiplo_atr = 1.5
        valorizacao_sugerida = ultimo_atr_pct * multiplo_atr
        # arredondar valorização sugerida para cima com duas casas
        valorizacao_sugerida = np.ceil(valorizacao_sugerida * 100) / 100
        stop = np.ceil(ultimo_atr_pct)  # stop loss sugerido como 1 x último ATR

        for d in range(1, 11):
            print(f'Simulando {d} dias de valorização futura...')
            dias_futuro = d
            df = pp.calcular_target_com_stop(df_base, valorizacao_alvo_pct=valorizacao_sugerida, dias_horizonte_alvo=d, stop_loss_pct=stop)
            
            if (df['Target'].value_counts(normalize=True).values[1] > 0.2):            
                print(f'Valorização futura de {dias_futuro} dias é viável.')
                break

        dias_futuro_final = d # Exemplo        
        df = pp.calcular_target_com_stop(df_base, valorizacao_alvo_pct=valorizacao_sugerida, dias_horizonte_alvo=dias_futuro_final, stop_loss_pct=stop)
    except Exception as e:
        print(f'Erro na preparação de dados para {papel}: {e}')
        continue
    
    for dias_janela_x in lista_janelas:    
        print(f'\nAnalisando {papel} com janela de {dias_janela_x} dias...')    
        
        X, y = pp.preparar_X_y_agregado(df, dias_janela_x, dropna=True)    
        if len(X) < 150: # Mínimo para TimeSeriesSplit(n_splits=5) ter um treino inicial razoável
            print(f"Dados insuficientes ({len(X)} linhas) após preparação. Pulando config.")
            continue
            
        X_train, y_train, X_validacao, y_validacao = tr.dividir_treino_validacao(X, y, dias_validacao=DIAS_VALIDACAO, dias_futuro=dias_futuro_final)

        # Modelo template - será clonado dentro da função de avaliação
        modelo_template = LogisticRegression(solver='liblinear', penalty='l2', random_state=42, class_weight='balanced')
        
        # --- AVALIAÇÃO ROBUSTA COM TimeSeriesSplit DENTRO DO CONJUNTO DE TREINO ---
        # metricas_wf_treino = avaliar_estrategia_com_timeseriessplit(
        #     modelo_template=modelo_template,
        #     X_super_treino=X_train,
        #     y_super_treino=y_train,
        #     n_splits=5 # <<< Definindo o número de folds desejado
        # )
        precisao_minima = 0.5
        metricas_robustas_treino = avaliar_estrategia_com_recall_em_precisao_minima(
            modelo_template=modelo_template,
            X_super_treino=X_train,
            y_super_treino=y_train,
            n_splits=5,
            precisao_minima_desejada=precisao_minima # <<< SEU OBJETIVO DE 60% DE PRECISÃO
        )

        df_experimentos.loc[i, 'recall_na_prec_min'] = metricas_robustas_treino['recall_medio_na_prec_min']
        df_experimentos.loc[i, 'recall_std_na_prec_min'] = metricas_robustas_treino['recall_std_na_prec_min']
        df_experimentos.loc[i, 'prec_min'] = precisao_minima
        df_experimentos.loc[i, 'folds_com_sinal_pct'] = metricas_robustas_treino['prob_sucesso_folds']
        df_experimentos.loc[i, 'papel'] = papel
        df_experimentos.loc[i, 'dias_futuro'] = dias_futuro
        df_experimentos.loc[i, 'dias_janela_x'] = dias_janela_x
        df_experimentos.loc[i, 'valorizacao'] = valorizacao_sugerida
        df_experimentos.loc[i, 'stop'] = stop       


        
        # --- LOG DOS EXPERIMENTOS ---
        i += 1
        experimento_atual = {
            'papel': papel,
            'dias_futuro': dias_futuro_final,
            'dias_janela_x': dias_janela_x,
            # 'ultimo_atr_pct': ultimo_atr_pct, # Adicione se precisar
            'valorizacao': valorizacao_sugerida,
            'stop': stop,
            'f1_wf_treino': f1_wf_treino,
            'f1_wf_treino_std': f1_wf_treino_std,
            'precisao_wf_treino': metricas_wf_treino['precisao_media'],
            'recall_wf_treino': metricas_wf_treino['recall_medio']
        }
        #df_experimentos = pd.concat([df_experimentos, pd.DataFrame([experimento_atual], index=[i])])

[ABEV3.SA] Cache existente cobre de 2020-01-02 a 2025-07-31.
[ABEV3.SA] Cache cobre período solicitado (2022-09-14 a 2025-07-30).
[ABEV3.SA] Garantindo atualização dos últimos 5 dias do cache...
Index(['data', 'adj_close', 'close', 'Dividends', 'high', 'low', 'open',
       'Stock Splits', 'volume'],
      dtype='object', name='Price')
[ABEV3.SA] Últimos dias atualizados em memória com sucesso.
[ABEV3.SA] Retornando 717 linhas para o período solicitado.
✅ Usando cache local: /palpiteiro/dados\macro_20210914_20250731.csv
[^BVSP] Cache existente cobre de 2020-01-02 a 2025-07-31.
[^BVSP] Cache cobre período solicitado (2022-09-14 a 2025-07-30).
[^BVSP] Garantindo atualização dos últimos 5 dias do cache...
Index(['data', 'adj_close', 'close', 'Dividends', 'high', 'low', 'open',
       'Stock Splits', 'volume'],
      dtype='object', name='Price')
[^BVSP] Últimos dias atualizados em memória com sucesso.
[^BVSP] Retornando 717 linhas para o período solicitado.
[BOVA11.SA] Cache existente cob


1 Failed download:
['BRDT3.SA']: YFTzMissingError('possibly delisted; no timezone found')


INFO: 21 linha(s) removidas de X e y devido a NaNs (dropna='linhas').

--- Avaliando Estratégia com Recall na Precisão Mínima de 50% (n_splits=5) ---
Fold 1/5: Treinando com 98, Validando com 30...
  Recall na Precisão >= 50%: 0.778
Fold 2/5: Treinando com 193, Validando com 30...
  Recall na Precisão >= 50%: 0.000
Fold 3/5: Treinando com 288, Validando com 30...
  Recall na Precisão >= 50%: 0.000
Fold 4/5: Treinando com 383, Validando com 30...
  Recall na Precisão >= 50%: 0.000
Fold 5/5: Treinando com 478, Validando com 30...
  Recall na Precisão >= 50%: 0.000

Resultado Final da Avaliação Walk-Forward:
  Recall Médio (com Precisão >= 50%): 0.156
  Desvio Padrão do Recall: 0.311
  Percentual de Folds com Sinal Viável: 20%
[BRDT3.SA] Cache não existe ou está vazio. Download completo necessário.
[BRDT3.SA] Iniciando download de dados (lógica de atualização/complemento)...
[BRDT3.SA] Período de download yfinance: 2020-01-01 a 2025-08-04
[BRDT3.SA] AVISO: Nenhum dado retornado pelo yfina


1 Failed download:
['BRML3.SA']: YFTzMissingError('possibly delisted; no timezone found')

1 Failed download:
['BTOW3.SA']: YFTzMissingError('possibly delisted; no timezone found')


INFO: 21 linha(s) removidas de X e y devido a NaNs (dropna='linhas').

--- Avaliando Estratégia com Recall na Precisão Mínima de 50% (n_splits=5) ---
Fold 1/5: Treinando com 98, Validando com 30...
  Recall na Precisão >= 50%: 1.000
Fold 2/5: Treinando com 193, Validando com 30...
  Recall na Precisão >= 50%: 0.000
Fold 3/5: Treinando com 288, Validando com 30...
  Recall na Precisão >= 50%: 0.000
Fold 4/5: Treinando com 383, Validando com 30...
  Recall na Precisão >= 50%: 0.000
Fold 5/5: Treinando com 478, Validando com 30...
  Recall na Precisão >= 50%: 0.000

Resultado Final da Avaliação Walk-Forward:
  Recall Médio (com Precisão >= 50%): 0.200
  Desvio Padrão do Recall: 0.400
  Percentual de Folds com Sinal Viável: 20%
[BRML3.SA] Cache não existe ou está vazio. Download completo necessário.
[BRML3.SA] Iniciando download de dados (lógica de atualização/complemento)...
[BRML3.SA] Período de download yfinance: 2020-01-01 a 2025-08-04
[BRML3.SA] AVISO: Nenhum dado retornado pelo yfina


1 Failed download:
['CCRO3.SA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2020-01-01 -> 2025-08-05) (Yahoo error = "No data found, symbol may be delisted")')


... (31 outputs left out)


1 Failed download:
['VIVT4.SA']: YFTzMissingError('possibly delisted; no timezone found')


[VIVT4.SA] AVISO: Nenhum dado retornado pelo yfinance para o download.
[VIVT4.SA] AVISO FINAL: Nenhum dado disponível para VIVT4.SA.
Erro na preparação de dados para VIVT4.SA: 'close'
[VVAR3.SA] Cache não existe ou está vazio. Download completo necessário.
[VVAR3.SA] Iniciando download de dados (lógica de atualização/complemento)...
[VVAR3.SA] Período de download yfinance: 2020-01-01 a 2025-08-04



1 Failed download:
['VVAR3.SA']: YFTzMissingError('possibly delisted; no timezone found')


[VVAR3.SA] AVISO: Nenhum dado retornado pelo yfinance para o download.
[VVAR3.SA] AVISO FINAL: Nenhum dado disponível para VVAR3.SA.
Erro na preparação de dados para VVAR3.SA: 'close'
[WEGE3.SA] Cache existente cobre de 2020-01-02 a 2025-07-14.
[WEGE3.SA] Cache NÃO cobre período solicitado. Download total necessário.
[WEGE3.SA] Iniciando download de dados (lógica de atualização/complemento)...
[WEGE3.SA] Período de download yfinance: 2020-01-01 a 2025-08-04
Index(['data', 'adj_close', 'close', 'Dividends', 'high', 'low', 'open',
       'Stock Splits', 'volume'],
      dtype='object', name='Price')
[WEGE3.SA] Realizando verificação de divergência entre cache antigo e dados frescos...
[WEGE3.SA] Pós-Download Total: Nenhuma divergência significativa nas amostras (tolerância 1.00%).
[WEGE3.SA] Cache principal ATUALIZADO com dados frescos: /palpiteiro/dados\WEGE3.csv
[WEGE3.SA] Retornando 717 linhas para o período solicitado.
✅ Usando cache local: /palpiteiro/dados\macro_20210914_20250731.c

In [6]:
#df_experimentos.to_csv('experimentos_wf_recall_prec_50.csv', index=False)
df_experimentos = pd.read_csv('experimentos_wf_recall_prec_50.csv')

In [7]:
#df_experimentos = pd.read_csv('experimentos_wf_recall_prec_50.csv')
df_experimentos['coef_variacao_recall'] = df_experimentos['recall_std_na_prec_min'] / df_experimentos['recall_na_prec_min']
df_experimentos['coef_variacao_recall'].describe()

count    286.000000
mean       1.346784
std        0.467288
min        0.327833
25%        0.942327
50%        1.261149
75%        1.948913
max        2.000000
Name: coef_variacao_recall, dtype: float64

In [8]:
df_experimentos['recall_na_prec_min'].describe()

count    295.000000
mean       0.266983
std        0.158238
min        0.000000
25%        0.150000
50%        0.250000
75%        0.378889
max        0.710000
Name: recall_na_prec_min, dtype: float64

In [9]:
df_experimentos[df_experimentos['recall_na_prec_min'] > 0.3]['coef_variacao_recall'].describe()

count    113.000000
mean       0.937859
std        0.229018
min        0.327833
25%        0.803055
50%        0.914943
75%        1.142229
max        1.298442
Name: coef_variacao_recall, dtype: float64

In [10]:
df_experimentos[df_experimentos['recall_na_prec_min'] > 0.3].sort_values(by='coef_variacao_recall', ascending=True).head(10)

,recall_na_prec_min,recall_std_na_prec_min,prec_min,folds_com_sinal_pct,papel,dias_futuro,dias_janela_x,valorizacao,stop,coef_variacao_recall
139,0.710000,0.232761,0.5,1.0,EZTC3.SA,3.0,30.0,0.05,1.0,0.327833
167,0.487302,0.168685,0.5,1.0,ITUB4.SA,3.0,15.0,0.03,1.0,0.346161
90,0.677273,0.267526,0.5,1.0,CVCB3.SA,2.0,6.0,0.07,1.0,0.395005
119,0.650216,0.344719,0.5,0.8,ELET6.SA,3.0,30.0,0.03,1.0,0.530161
11,0.683333,0.388730,0.5,1.0,BBAS3.SA,6.0,10.0,0.04,1.0,0.568873
12,0.537179,0.312789,0.5,1.0,BBAS3.SA,6.0,15.0,0.04,1.0,0.582280
221,0.592624,0.348229,0.5,0.8,QUAL3.SA,2.0,10.0,0.06,1.0,0.587605
60,0.597619,0.355265,0.5,0.8,CMIG4.SA,4.0,6.0,0.04,1.0,0.594467
174,0.680000,0.411825,0.5,0.8,KLBN11.SA,5.0,30.0,0.04,1.0,0.605625
232,0.485714,0.295699,0.5,0.8,RAIL3.SA,4.0,15.0,0.05,1.0,0.608791


In [16]:
df_experimentos['recall_std_na_prec_min'].describe()

count    295.000000
mean       0.304680
std        0.126499
min        0.000000
25%        0.224696
50%        0.350000
75%        0.390620
max        0.489898
Name: recall_std_na_prec_min, dtype: float64

In [17]:
df_experimentos.sort_values(by='recall_na_prec_min', ascending=False).head(20)

,recall_na_prec_min,recall_std_na_prec_min,prec_min,folds_com_sinal_pct,papel,dias_futuro,dias_janela_x,valorizacao,stop,coef_variacao_recall
139,0.710000,0.232761,0.5,1.0,EZTC3.SA,3.0,30.0,0.05,1.0,0.327833
11,0.683333,0.388730,0.5,1.0,BBAS3.SA,6.0,10.0,0.04,1.0,0.568873
174,0.680000,0.411825,0.5,0.8,KLBN11.SA,5.0,30.0,0.04,1.0,0.605625
90,0.677273,0.267526,0.5,1.0,CVCB3.SA,2.0,6.0,0.07,1.0,0.395005
119,0.650216,0.344719,0.5,0.8,ELET6.SA,3.0,30.0,0.03,1.0,0.530161
117,0.619048,0.377364,0.5,0.8,ELET6.SA,3.0,15.0,0.03,1.0,0.609588
245,0.600000,0.489898,0.5,0.6,SBSP3.SA,4.0,6.0,0.04,1.0,0.816497
60,0.597619,0.355265,0.5,0.8,CMIG4.SA,4.0,6.0,0.04,1.0,0.594467
221,0.592624,0.348229,0.5,0.8,QUAL3.SA,2.0,10.0,0.06,1.0,0.587605
205,0.586667,0.387356,0.5,0.8,PETR3.SA,3.0,6.0,0.04,1.0,0.660265
